###### Path to notebook: [https://www.github.com/microsoft/presidio/blob/main/docs/samples/python/presidio_notebook.ipynb](https://www.github.com/microsoft/presidio/blob/main/docs/samples/python/presidio_notebook.ipynb)

In [1]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import json
from pprint import pprint

# Analyze Text for PII Entities

Using Presidio Analyzer, analyze a text to identify PII entities. 
The Presidio analyzer is using pre-defined entity recognizers, and offers the option to create custom recognizers.

The following code sample will:

- Set up the Analyzer engine: load the NLP module (spaCy model by default) and other PII recognizers
- Call analyzer to get analyzed results for "PHONE_NUMBER" entity type


In [13]:
text_to_anonymize = """
His name is 
First Name: Jone
Last Name: Davis
and his phone number is 212-555-5555

123 MAIN ST ddd ddd ddd
Maple, MA 212444
"""

In [3]:
analyzer = AnalyzerEngine()
analyzer_results = analyzer.analyze(text=text_to_anonymize, entities=["PHONE_NUMBER"], language='en')

print(analyzer_results)

[type: PHONE_NUMBER, start: 46, end: 58, score: 0.75]


## Create Custom PII Entity Recognizers

Presidio Analyzer comes with a pre-defined set of entity recognizers. It also allows adding new recognizers without changing the analyzer base code, **by creating custom recognizers**. 
In the following example, we will create two new recognizers of type `PatternRecognizer` to identify titles and pronouns in the analyzed text.
A `PatternRecognizer` is a PII entity recognizer which uses regular expressions or deny-lists.

The following code sample will:
- Create custom recognizers
- Add the new custom recognizers to the analyzer
- Call analyzer to get results from the new recognizers

In [4]:
titles_recognizer = PatternRecognizer(supported_entity="TITLE",
                                      deny_list=["Mr.","Mrs.","Miss"])

pronoun_recognizer = PatternRecognizer(supported_entity="PRONOUN",
                                       deny_list=["he", "He", "his", "His", "she", "She", "hers", "Hers"])

analyzer.registry.add_recognizer(titles_recognizer)
analyzer.registry.add_recognizer(pronoun_recognizer)

analyzer_results = analyzer.analyze(text=text_to_anonymize,
                            entities=["TITLE", "PRONOUN"],
                            language="en")
print(analyzer_results)


[type: PRONOUN, start: 0, end: 3, score: 1.0, type: TITLE, start: 12, end: 15, score: 1.0, type: PRONOUN, start: 26, end: 29, score: 1.0]


Call Presidio Analyzer and get analyzed results with all the configured recognizers - default and new custom recognizers

In [14]:
analyzer_results = analyzer.analyze(text=text_to_anonymize, language='en')

analyzer_results

[type: PRONOUN, start: 1, end: 4, score: 1.0,
 type: PRONOUN, start: 52, end: 55, score: 1.0,
 type: PERSON, start: 26, end: 31, score: 0.85,
 type: PERSON, start: 42, end: 47, score: 0.85,
 type: PHONE_NUMBER, start: 72, end: 84, score: 0.75,
 type: US_DRIVER_LICENSE, start: 120, end: 126, score: 0.01]

# Anonymize Text with Identified PII Entities

<br>Presidio Anonymizer iterates over the Presidio Analyzer result, and provides anonymization capabilities for the identified text.
<br>The anonymizer provides 5 types of anonymizers - replace, redact, mask, hash and encrypt. The default is **replace**

<br>The following code sample will:
<ol>
<li>Setup the anonymizer engine </li>
<li>Create an anonymizer request - text to anonymize, list of anonymizers to apply and the results from the analyzer request</li>
<li>Anonymize the text</li>
</ol>

In [6]:
anonymizer = AnonymizerEngine()

anonymized_results = anonymizer.anonymize(
    text=text_to_anonymize,
    analyzer_results=analyzer_results,    
    operators={"DEFAULT": OperatorConfig("replace", {"new_value": "<ANONYMIZED>"}), 
                        "PHONE_NUMBER": OperatorConfig("mask", {"type": "mask", "masking_char" : "*", "chars_to_mask" : 12, "from_end" : True}),
                        "TITLE": OperatorConfig("redact", {})}
)

print(f"text: {anonymized_results.text}")
print("detailed response:")

pprint(json.loads(anonymized_results.to_json()))

text: <ANONYMIZED> name is  <ANONYMIZED> and <ANONYMIZED> phone number is ************
detailed response:
{'items': [{'end': 80,
            'entity_type': 'PHONE_NUMBER',
            'operator': 'mask',
            'start': 68,
            'text': '************'},
           {'end': 51,
            'entity_type': 'PRONOUN',
            'operator': 'replace',
            'start': 39,
            'text': '<ANONYMIZED>'},
           {'end': 34,
            'entity_type': 'PERSON',
            'operator': 'replace',
            'start': 22,
            'text': '<ANONYMIZED>'},
           {'end': 21,
            'entity_type': 'TITLE',
            'operator': 'redact',
            'start': 21,
            'text': ''},
           {'end': 12,
            'entity_type': 'PRONOUN',
            'operator': 'replace',
            'start': 0,
            'text': '<ANONYMIZED>'}],
 'text': '<ANONYMIZED> name is  <ANONYMIZED> and <ANONYMIZED> phone number is '
         '************'}
